# 2.6 학습 목표

PBRFT는 즉각 보상 E[R(s,a)]를 최적화하고,
Agentic RL은 할인 누적 반환 E[Σ γ^t R_t]를 최적화한다.

### PBRFT 목표
J(θ) = E[R(s,a)] - 시간적 의존성 없음

### Agentic RL 목표  
J(θ) = E[Σ_{t=0}^T γ^t R_t] - 할인된 누적 보상

여기서 γ ∈ [0,1]은 할인 인자다.
- γ → 0: 즉각 보상 중시 (근시안적)
- γ → 1: 미래 보상 중시 (원시안적)
- γ ≈ 0.99: 균형 있는 선택

In [1]:
from utils_openai import *
import numpy as np

## 실습 1: PBRFT 목표 - 즉각 보상 최적화

In [2]:
heading("PBRFT: J = E[R(s,a)]")

prompts = ["2+2는?", "3+3은?", "4+4는?"]
rewards = []

for prompt in prompts:
    response = ask(prompt, temperature=0.0)
    score = 10.0 if any(str(i) in response for i in range(10)) else 0.0
    rewards.append(score)
    step_print(len(rewards), prompt, f"보상: {score:.1f}")

expected_reward = np.mean(rewards)
kv_print([("목표 J(θ) = E[R]", f"{expected_reward:.1f}")])
print("✓ 각 응답의 즉각 보상의 평균을 최대화한다.")



────────────────────────────────────────
  PBRFT: J = E[R(s,a)]
────────────────────────────────────────
  [Step 1] 2+2는?
    보상: 10.0
  [Step 2] 3+3은?
    보상: 10.0
  [Step 3] 4+4는?
    보상: 10.0
  목표 J(θ) = E[R]  10.0
✓ 각 응답의 즉각 보상의 평균을 최대화한다.


## 실습 2: Agentic RL 목표 - 할인 누적 보상

In [4]:
heading("Agentic RL: J = E[Σ γ^t R_t]")

rewards_sequence = [1.0, 2.0, 5.0, 10.0]  # 각 단계의 보상
gamma = 0.9

# 할인 누적 계산
discounted_sum = sum(r * (gamma ** i) for i, r in enumerate(rewards_sequence))

print("단계별 보상과 할인:")
for i, r in enumerate(rewards_sequence):
    discounted = r * (gamma ** i)
    step_print(i+1, f"Step {i}", f"R={r} × γ^{i} = {discounted:.3f}")

kv_print([("목표 J(θ) = Σ γ^t R_t", f"{discounted_sum:.2f}")])
print("✓ 미래 보상도 고려하되, 시간에 따라 지수적으로 감쇠한다.")


────────────────────────────────────────
  Agentic RL: J = E[Σ γ^t R_t]
────────────────────────────────────────
단계별 보상과 할인:
  [Step 1] Step 0
    R=1.0 × γ^0 = 1.000
  [Step 2] Step 1
    R=2.0 × γ^1 = 1.800
  [Step 3] Step 2
    R=5.0 × γ^2 = 4.050
  [Step 4] Step 3
    R=10.0 × γ^3 = 7.290
  목표 J(θ) = Σ γ^t R_t  14.14
✓ 미래 보상도 고려하되, 시간에 따라 지수적으로 감쇠한다.


## 실습 3: 할인 인자의 영향

In [6]:
heading("할인 인자 γ의 영향")

rewards = [1.0, 1.0, 1.0, 100.0]  # 마지막에 큰 보상
gammas = [0.5, 0.9, 0.99, 1.0]

print("다양한 γ 값에서의 누적 보상:")
for gamma in gammas:
    total = sum(r * (gamma ** i) for i, r in enumerate(rewards))
    future_weight = (100.0 * gamma**3) / total * 100 if total > 0 else 0
    print(f"  γ={gamma:.2f}: G₀ = {total:.1f} (마지막 보상 기여도 {future_weight:.0f}%)")

print("해석:")
print("  - γ=0.5: 과거 보상만 중요 (단기적)")
print("  - γ=0.99: 미래도 중요 (장기적)")


────────────────────────────────────────
  할인 인자 γ의 영향
────────────────────────────────────────
다양한 γ 값에서의 누적 보상:
  γ=0.50: G₀ = 14.2 (마지막 보상 기여도 88%)
  γ=0.90: G₀ = 75.6 (마지막 보상 기여도 96%)
  γ=0.99: G₀ = 100.0 (마지막 보상 기여도 97%)
  γ=1.00: G₀ = 103.0 (마지막 보상 기여도 97%)
해석:
  - γ=0.5: 과거 보상만 중요 (단기적)
  - γ=0.99: 미래도 중요 (장기적)
